# FreshMart Lab 1: Exploratory Data Analysis
**Microsoft Fabric Data Science**

บทบาท: **Analyst** — สำรวจข้อมูลก่อนสร้างโมเดล  
จด **3 ข้อสังเกตสั้น ๆ** ส่ง Lab 2 (ไม่ต้องจำสูตรสถิติ)

### สิ่งที่แล็บนี้ตอบ
1. ข้อมูลขาดตรงไหน?
2. ของเสียสูงที่ประเภทสาขาไหน?
3. คนที่ Churn พฤติกรรมต่างจากคนอยู่ต่ออย่างไร?

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ความหมาย |
| --- | --- |
| กราฟไม่ขึ้น | รอเซลล์ก่อนหน้าจบ แล้วรันใหม่ |
| ค่าสหสัมพันธ์ไม่ตรงทศนิยมทุกตัว | ปัด 2 ตำแหน่งใกล้เคียงพอ |
| แนบ lakehouse ยังไม่ได้ | กลับไป Lab 0 |


### เตรียมตัวโหลดข้อมูล

รันเซลล์ถัดไปเพื่อนิยาม `load_table_or_csv` (เหมือน Lab 0) — **อย่าข้าม**


In [ ]:
from pathlib import Path
import pandas as pd

# Schema-qualified names (preferred). Legacy flat names still tried as fallback.
BRONZE_TRANSACTIONS = "bronze.transactions"
BRONZE_CUSTOMERS = "bronze.customers"
SILVER_CUSTOMER_FEATURES = "silver.customer_features"
GOLD_PREDICTIONS = "gold.freshmart_predictions"

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def _table_candidates(table_name: str) -> list[str]:
    legacy = {
        "bronze.transactions": "bronze_transactions",
        "bronze.customers": "bronze_customers",
        "silver.customer_features": "silver_customer_features",
        "gold.freshmart_predictions": "gold_freshmart_predictions",
    }
    names = [table_name]
    if table_name in legacy:
        names.append(legacy[table_name])
    return names

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    last_error = None
    for candidate in _table_candidates(table_name):
        try:
            frame = spark.read.table(candidate).toPandas()
            print(f"Loaded Spark table {candidate}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            last_error = exc
    print(f"Spark table '{table_name}' unavailable ({last_error}). Falling back to CSV.")
    return load_csv(file_name)


### ขั้นตอนที่ 1: โหลดธุรกรรม

**โค้ดนี้ทำอะไร:** อ่าน `bronze.transactions` แล้วแปลงเป็น Pandas

- `shape` = `(จำนวนแถว, จำนวนคอลัมน์)` → ต้องได้ประมาณ `(3000, 14)`
- `head()` = ดู 5 แถวแรก เพื่อรู้จักคอลัมน์ เช่น `WasteUnits`, `StoreType`


In [ ]:
df = load_table_or_csv("bronze.transactions", "freshmart_transactions.csv")
print(f"Pandas DataFrame shape: {df.shape}")
df.head()


### ขั้นตอนที่ 2: คุณภาพข้อมูล (ค่าว่าง)

**ความรู้จำเป็น**
- ค่าว่าง (missing) ทำให้โมเดล/สถิติเพี้ยน — ต้องรู้ก่อนแก้ใน Lab 2
- `isnull().sum()` นับแถวว่างต่อคอลัมน์

**ต้องเห็น:** `DiscountRate` ว่างประมาณ **89 แถว (~3%)**  
คอลัมน์อื่นไม่ควรว่างจำนวนมาก — ถ้าเป็นแบบนั้น แจ้ง TA


In [ ]:
print("=== DataFrame Info ===")
df.info()
print("\n=== Missing Values Count ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"DiscountRate missing rate: {df['DiscountRate'].isna().mean():.2%}")


### ขั้นตอนที่ 3: สถิติเชิงพรรณนา

**โค้ดนี้ทำอะไร:** `describe()` สรุป mean / min / max / เปอร์เซ็นไทล์

ดูคอลัมน์ `WasteUnits`: ค่าส่วนใหญ่ต่ำ แต่มีหางยาว (= เบ้ขวา) — ของเสียกระจุกบางวัน/บางสาขา


In [ ]:
df[["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost"]].describe()


### ขั้นตอนที่ 4: Histogram — เห็นการกระจาย

**ความรู้จำเป็น**
- Histogram = นับความถี่ตามช่วงค่า
- เส้นโค้งบนกราฟช่วยดูรูปแบบการกระจาย

**ต้องเห็น:** `WasteUnits` เบ้ขวา (ค่าสูงมีน้อย)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["UnitsSold"], bins=15, kde=True, color="#00D2B4", ax=axes[0])
axes[0].set_title("Distribution of Units Sold")
sns.histplot(df["WasteUnits"], bins=10, kde=True, color="#F43F5E", ax=axes[1])
axes[1].set_title("Distribution of Waste Units (right-skewed)")
plt.tight_layout()
plt.show()


### Box plot ตามประเภทสาขา

**ความรู้จำเป็น:** Box plot เปรียบการกระจายระหว่างกลุ่มได้เร็ว

**อินไซต์ที่คาดหวัง:** Express ของเสียสูงกว่า Hypermarket โดยประมาณ  
(พื้นที่จัดเก็บจำกัด / ของเสียช่วงสุดสัปดาห์)

ไม่ต้องได้ตัวเลขเป๊ะทุกทศนิยม — เห็นแนวโน้มถูกทางพอ


In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="StoreType", y="WasteUnits", hue="StoreType", palette="Set2", legend=False)
plt.title("Waste Units by Store Type")
plt.show()


### ขั้นตอนที่ 5: Correlation

**ความรู้จำเป็น**
- ค่าใกล้ **+1** = ไปด้วยกัน, ใกล้ **-1** = สวนทาง, ใกล้ **0** = เกือบไม่เกี่ยว
- Heatmap คือตารางสหสัมพันธ์แบบสี

**ค่าอ้างอิงชุดนี้ (ปัด 2 ตำแหน่ง)**
- DiscountRate ↔ UnitsSold ≈ **+0.38** (ลดราคาแล้วขายดีขึ้น)
- DiscountRate ↔ WasteUnits ≈ **-0.12** (ลดราคามักเหลือทิ้งน้อยลง)


In [ ]:
numeric_cols = ["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost", "IsWeekend"]
corr_matrix = df[numeric_cols].corr(numeric_only=True)
print(corr_matrix.round(2))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, linewidths=0.5)
plt.title("FreshMart Feature Correlation Matrix")
plt.show()


### ขั้นตอนที่ 6: สำรวจ Churn ของสมาชิก

**ความรู้จำเป็น**
- `Churn = 1` = มีแนวโน้มยกเลิก, `0` = อยู่ต่อ
- อัตรา Churn ชุดนี้ ≈ **19.3%** (290 จาก 1,500)
- `Age` ว่าง **37** แถว → Lab 2 จะเติมด้วย median

Scatter ด้านล่าง: คนขาดซื้อนาน (`RecencyDays` สูง) + ร้องเรียนบ่อย มักกระจุกที่ Churn=1


In [ ]:
df_cust = load_table_or_csv("bronze.customers", "freshmart_customers.csv")
print(f"Total Customers: {len(df_cust):,}")
print(df_cust["Churn"].value_counts(normalize=True).rename("rate"))
print(f"Age missing: {df_cust['Age'].isna().sum()} ({df_cust['Age'].isna().mean():.2%})")

plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df_cust,
    x="RecencyDays",
    y="ComplaintCount",
    hue="Churn",
    palette={0: "#00D2B4", 1: "#F43F5E"},
    alpha=0.7,
)
plt.title("Customer Churn Pattern: Recency vs Complaint Count")
plt.show()


### จุดตรวจ Lab 1

ผ่านแล้วพิมพ์ `Lab 1 verification passed`  
ก่อนไป Lab 2 จด 3 ข้อ: **เติม Age · แปลงหมวดหมู่ · ปรับสเกลเงิน/ความถี่**


In [ ]:
if len(df) != 3000:
    raise AssertionError(f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df):,}")
if len(df_cust) != 1500:
    raise AssertionError(f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,}")
if df["DiscountRate"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี DiscountRate ว่าง เพื่อฝึกจัดการค่าว่างใน Lab 2")
if df_cust["Age"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี Age ว่าง เพื่อฝึกเติมค่าใน Lab 2")
print("Lab 1 verification passed")
